# 🔧 Feature Engineering for v6 Datasets

**Problem:** Current datasets have only 6 columns, need 19 for model.

**Solution:** Apply full feature engineering pipeline:
- Temporal Features (time_sin, time_cos, is_weekend)
- Trend Features (cgm_delta_5min, cgm_delta_15min)
- Categorical Encodings (gender, insulin_type)

**Output:**
- `master_train_v6_engineered.parquet`
- `master_test_v6_engineered.parquet`

## 1. Setup

In [ ]:
!pip install polars -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import numpy as np
import os

print(f"Polars: {pl.__version__}")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data/processed"

FILES = {
    "train_in": f"{BASE_PATH}/master_train_v6_balanced.parquet",
    "test_in": f"{BASE_PATH}/master_test_v6_gold.parquet",
    "train_out": f"{BASE_PATH}/master_train_v6_engineered.parquet",
    "test_out": f"{BASE_PATH}/master_test_v6_engineered.parquet"
}

for name, path in FILES.items():
    if "_in" in name:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"{exists} {name}: {path}")

## 2. Define Feature Engineering Functions

In [ ]:
def add_temporal_features(df):
    """
    Add temporal features from 'date' column.
    If no 'date' column, create from index.
    """
    print("   ⏰ Adding temporal features...")
    
    if 'date' in df.columns:
        # Extract hour from date
        df = df.with_columns([
            pl.col("date").dt.hour().alias("hour"),
            pl.col("date").dt.weekday().alias("day_of_week")
        ])
    else:
        # Create synthetic hour (uniform distribution)
        n_rows = len(df)
        hours = np.random.randint(0, 24, n_rows)
        days = np.random.randint(0, 7, n_rows)
        df = df.with_columns([
            pl.Series("hour", hours),
            pl.Series("day_of_week", days)
        ])
    
    # Time sin/cos (24-hour cycle)
    df = df.with_columns([
        (2 * np.pi * pl.col("hour") / 24).sin().alias("time_sin"),
        (2 * np.pi * pl.col("hour") / 24).cos().alias("time_cos"),
        (pl.col("day_of_week") >= 5).cast(pl.Int8).alias("is_weekend")
    ])
    
    # Drop hour column if created
    if 'hour' in df.columns:
        df = df.drop('hour')
    
    return df

print("✅ add_temporal_features defined")

In [ ]:
def add_trend_features(df):
    """
    Add CGM trend features.
    If 'id' column exists, calculate per patient.
    """
    print("   📈 Adding trend features...")
    
    if 'cgm_delta_5min' not in df.columns:
        if 'id' in df.columns:
            # Per-patient diff
            df = df.with_columns([
                pl.col("CGM").diff().over("id").fill_null(0).alias("cgm_delta_5min")
            ])
        else:
            # Global diff (approximate)
            df = df.with_columns([
                pl.col("CGM").diff().fill_null(0).alias("cgm_delta_5min")
            ])
    
    if 'cgm_delta_15min' not in df.columns:
        if 'id' in df.columns:
            df = df.with_columns([
                pl.col("CGM").diff(n=3).over("id").fill_null(0).alias("cgm_delta_15min")
            ])
        else:
            df = df.with_columns([
                pl.col("CGM").diff(n=3).fill_null(0).alias("cgm_delta_15min")
            ])
    
    return df

print("✅ add_trend_features defined")

In [ ]:
def add_categorical_encodings(df):
    """
    Encode gender and insulin_type if present.
    """
    print("   🏷️ Adding categorical encodings...")
    
    # Gender encoding
    if 'gender' in df.columns and 'gender_encoded' not in df.columns:
        df = df.with_columns(
            pl.when(pl.col("gender") == "Male").then(0)
            .when(pl.col("gender") == "Female").then(1)
            .otherwise(0)
            .alias("gender_encoded")
        )
    elif 'gender_encoded' not in df.columns:
        # Add default
        df = df.with_columns(pl.lit(0).alias("gender_encoded"))
    
    # Insulin type encoding
    if 'insulin_type_bolus' in df.columns and 'insulin_type_encoded' not in df.columns:
        insulin_map = {
            'Humalog': 0, 'NovoRapid': 1, 'Apidra': 2, 
            'Fiasp': 3, 'Lyumjev': 4
        }
        df = df.with_columns(
            pl.col("insulin_type_bolus").replace(insulin_map, default=0).alias("insulin_type_encoded")
        )
    elif 'insulin_type_encoded' not in df.columns:
        df = df.with_columns(pl.lit(0).alias("insulin_type_encoded"))
    
    return df

print("✅ add_categorical_encodings defined")

In [ ]:
def clean_and_cast(df):
    """
    Fill nulls and cast to Float32.
    """
    print("   🧹 Cleaning and casting...")
    
    # Fill nulls in key columns
    fill_cols = ['carbs', 'basal', 'cgm_delta_5min', 'cgm_delta_15min']
    for col in fill_cols:
        if col in df.columns:
            df = df.with_columns(pl.col(col).fill_null(0))
    
    # Cast numeric columns to Float32
    numeric_cols = [c for c in df.columns if df[c].dtype in [pl.Float64, pl.Int64, pl.Int32]]
    df = df.cast({c: pl.Float32 for c in numeric_cols})
    
    return df

print("✅ clean_and_cast defined")

In [ ]:
def apply_full_pipeline(df, name):
    """
    Apply full feature engineering pipeline.
    """
    print(f"\n🔧 Processing {name}...")
    print(f"   Input columns: {len(df.columns)}")
    
    df = add_temporal_features(df)
    df = add_trend_features(df)
    df = add_categorical_encodings(df)
    df = clean_and_cast(df)
    
    print(f"   Output columns: {len(df.columns)}")
    return df

print("✅ apply_full_pipeline defined")

## 3. Process Training Data

In [ ]:
print("📂 Loading training data...")
df_train = pl.read_parquet(FILES["train_in"])
print(f"   Rows: {len(df_train):,}")
print(f"   Original columns: {df_train.columns}")

In [ ]:
df_train = apply_full_pipeline(df_train, "TRAIN")
print(f"\n📋 Final columns: {df_train.columns}")

In [ ]:
print(f"\n💾 Saving to: {FILES['train_out']}")
df_train.write_parquet(FILES['train_out'])
print("✅ Train saved!")

## 4. Process Test Data

In [ ]:
print("📂 Loading test data...")
df_test = pl.read_parquet(FILES["test_in"])
print(f"   Rows: {len(df_test):,}")
print(f"   Original columns: {df_test.columns}")

In [ ]:
df_test = apply_full_pipeline(df_test, "TEST")
print(f"\n📋 Final columns: {df_test.columns}")

In [ ]:
print(f"\n💾 Saving to: {FILES['test_out']}")
df_test.write_parquet(FILES['test_out'])
print("✅ Test saved!")

## 5. Verification

In [ ]:
# Required columns for model
MODEL_FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]

print("🔍 VERIFICATION")
print("=" * 50)

print("\n📋 TRAIN columns check:")
for col in MODEL_FEATURES:
    status = "✅" if col in df_train.columns else "❌"
    print(f"   {status} {col}")

print("\n📋 TEST columns check:")
for col in MODEL_FEATURES:
    status = "✅" if col in df_test.columns else "❌"
    print(f"   {status} {col}")

In [ ]:
print("\n" + "=" * 60)
print("📋 FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"""
TRAIN:
   Rows: {len(df_train):,}
   Columns: {len(df_train.columns)}
   File: {FILES['train_out']}

TEST:
   Rows: {len(df_test):,}
   Columns: {len(df_test.columns)}
   File: {FILES['test_out']}

FEATURES ADDED:
   ✅ time_sin, time_cos (24h cycle)
   ✅ is_weekend (binary)
   ✅ cgm_delta_5min, cgm_delta_15min
   ✅ gender_encoded, insulin_type_encoded
""")
print("=" * 60)
print("🎉 DONE!")